In [20]:
import os
import sys
import numpy as np
import pandas as pd
import jax.numpy as jp
import scipy.sparse
import synth_pat.scripts.gast_model as gm

from synth_pat.paths import Paths
from synth_pat.scripts.simulation_utils import run_bold_sweep

subject_id = 'sub-2015060902'#sys.argv[1]
param_idx = 1#int(sys.argv[2])
output_file = './'#sys.argv[4]

med_dic = pd.read_csv(f'{Paths.RESOURCES}/mediction_sweep_params.csv')

Z_D1 = jp.array(med_dic['Z_D1'].values)
Z_D2 = jp.array(med_dic['Z_D2'].values)
Z_S  = jp.array(med_dic['Z_S'].values)

med_params = [Z_D1, Z_D2, Z_S]

type_of_sim = Paths.TYPE_OF_SWEEP

print(f"Running subject: {subject_id}")

DERIV_DIR = os.path.join(Paths.DATA, "derivatives", subject_id)

best_params_dic = pd.read_csv(f'{DERIV_DIR}/best_ppc_params_for_medication.csv')
njdopa_ctx_est = best_params_dic.loc[param_idx, 'njdopa_ctx']
njdopa_str_est = best_params_dic.loc[param_idx, 'njdopa_str']
ws_est = best_params_dic.loc[param_idx, 'ws']

# ------------------------
# Load subject-specific data
# ------------------------

L = pd.read_csv(os.path.join(DERIV_DIR, "dk_lengths_with_sero_and_dopa.csv"), index_col=0)
regions_names = L.columns.to_list()

Ceids = np.load(os.path.join(DERIV_DIR, "Ceids.npy"))
idelays = np.load(os.path.join(DERIV_DIR, "idelays.npy"))
Ja = np.load(os.path.join(DERIV_DIR, "Ja.npy"))
Rd1, Rd2, Rsero = np.load(os.path.join(DERIV_DIR, "Receptors.npy"))

# ------------------------
# Model setup
# ------------------------

setup = {
    "Seids": scipy.sparse.csr_matrix(Ceids),
    "idelays": idelays,
    "params": gm.sigm_d1d2sero_default_theta,
    "v_c": 3.9,
    "horizon": 650,
    "num_item": Z_D1.shape[0],
    "dt": 0.1,
    "num_skip": 10,
    "num_time": 300000,
    "init_state": jp.array([.01, -55.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]).reshape(10,1),
    "noise": 0.16,
}

# ------------------------
# Dopamine scaling
# ------------------------

JJdopa = np.ones(len(regions_names)) * njdopa_ctx_est

for region in ['L.PU', 'R.PU', 'L.CA', 'R.CA', 'L.PA', 'R.PA', 'L.AC', 'R.AC']:
    idx = regions_names.index(region)
    JJdopa[idx] = njdopa_str_est

JJdopa = JJdopa[:, None]

# ------------------------
# Parameter update
# ------------------------

theta = gm.sigm_d1d2sero_default_theta._replace(
    I=46.5,
    Ja=Ja,
    Jsa=Ja,
    Jsg=13,
    Jg=0,
    Jdopa=100000 * JJdopa,
    Rd1=Rd1,
    Rd2=Rd2,
    Rs=Rsero,
    Sd1=-10.0,
    Sd2=-10.0,
    Ss=-40.0,
    Zd1=Z_D1,
    Zd2=Z_D2,
    Zs=Z_S,
    we=0.3,
    wi=1.,
    wd=1,
    ws=ws_est,
    sigma_V=setup["noise"],
    sigma_u=0.1 * setup["noise"],
)

setup["params"] = theta

bold = run_bold_sweep((theta, setup))
bold = np.asarray(bold)

np.savez(output_file, bold=bold, params=med_params, med_params_names=['Z_D1', 'Z_D2', 'Z_S'], est_params=best_params_dic.values.astype('float'), est_params_names=best_params_dic.columns.to_list())

print(f"{subject_id} Done")

Running subject: sub-2015060902
sub-2015060902 Done


In [26]:
a = np.load('a.npz')

In [28]:
a['medication_name']

array('clozapine', dtype='<U9')

In [30]:
med_dic.loc[67]

medication    olanzapine
k               0.184211
k_i                    7
Z_D1             0.49658
Z_D2            0.673449
Z_S             0.157895
Name: 67, dtype: object